# Sample-Based Quantum Diagonalization: A Beginner-Friendly Workflow

This notebook is written as if we were sitting together at a whiteboard and building the idea slowly.

The central question is:

$$
H \lvert \psi \rangle = E \lvert \psi \rangle
$$

where:

- $H$ is the Hamiltonian matrix,
- $E$ is an energy eigenvalue,
- $\lvert \psi \rangle$ is an eigenstate.

In a large quantum problem, directly diagonalizing $H$ can be too expensive. Sample-based quantum diagonalization (SQD) tries to avoid that by using samples to identify a much smaller subspace that still contains the important physics.

In this notebook we will work through the full logic step by step, using a small toy Hamiltonian so every piece stays visible.

By the end of the notebook, you should be able to answer all of these questions clearly:

- What problem are we solving when we diagonalize a Hamiltonian?
- Why do basis states and measurement samples matter?
- How do we go from raw bitstrings to a reduced matrix problem?
- Why can a smaller projected eigenvalue problem still give a good approximation?

You do **not** need to be an expert in quantum computing before starting. The notebook explains the ideas from first principles and keeps reminding you what each symbol and each code cell is doing.

## Step 1: Know the roadmap before touching the code

Here is the plan we will follow:

1. Define a small Hamiltonian $H$.
2. Solve it exactly so we know the correct answer.
3. Convert the exact ground state into measurement probabilities.
4. Draw samples from those probabilities to imitate repeated measurements.
5. Use the most important sampled bitstrings to define a reduced subspace.
6. Project $H$ into that subspace.
7. Solve the smaller projected eigenvalue problem.
8. Compare the SQD estimate with the exact result.

Mathematically, the reduced problem is

$$
H_{\mathrm{sub}} c = E S c,
$$

where $H_{\mathrm{sub}}$ is the projected Hamiltonian and $S$ is the overlap matrix. In our toy example the chosen basis states are orthonormal, so $S$ will turn out to be the identity matrix, but it is useful to keep the general formula in view.

## A quick notation primer before we compute anything

If Dirac notation is new to you, here is the minimum you need for this notebook:

- $\lvert x \rangle$ means a basis state. In this notebook, $x$ is usually a three-bit string like $011$.
- A state such as $\lvert \psi \rangle$ is a vector made from a weighted combination of basis states.
- The weights in that combination are called amplitudes.
- Squaring the magnitude of an amplitude gives a measurement probability.

In ordinary linear algebra language, you can think of this notebook as solving a matrix eigenvalue problem and then approximating it inside a carefully chosen smaller vector space.

That means you can read the notebook in two equivalent ways:

- as a quantum story about states and measurements,
- or as a numerical linear algebra story about vectors, probabilities, and projections.

## Step 2: Import the tools we need

We keep the software stack intentionally small:

- `numpy` for arrays and linear algebra bookkeeping,
- `scipy.linalg.eigh` for Hermitian eigenvalue problems,
- `pandas` for neat tables,
- `matplotlib` for a few simple plots.

The goal of this notebook is to make the algorithm transparent, not to hide it behind a large framework.

In other words, every library here has a simple job:

- `numpy` stores the vectors and matrices,
- `scipy` solves the eigenvalue problems,
- `pandas` lets us inspect results in a readable table,
- `matplotlib` helps us see convergence instead of only reading numbers.

The code cell also fixes a random seed. That matters because the sampling stage is stochastic. Using a seed means you and I can reproduce the same example and talk about the same outputs.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.linalg import eigh

np.set_printoptions(precision=4, suppress=True)
plt.style.use("seaborn-v0_8-whitegrid")

SEED = 7
rng = np.random.default_rng(SEED)

## Step 3: Define the Hamiltonian and the computational basis

We use a three-qubit toy Hamiltonian, so the Hilbert space has dimension

$$
2^3 = 8.
$$

That means the computational basis is

$$
\{
\lvert 000 \rangle,
\lvert 001 \rangle,
\lvert 010 \rangle,
\lvert 011 \rangle,
\lvert 100 \rangle,
\lvert 101 \rangle,
\lvert 110 \rangle,
\lvert 111 \rangle
\}.
$$

The matrix below is small enough to inspect directly, but rich enough to show the SQD idea clearly.

A useful beginner mindset is this:

- each **row** and **column** corresponds to one basis state,
- each matrix entry tells us how those basis states are energetically related,
- diagonal entries describe energies tied to individual basis states,
- off-diagonal entries describe couplings or mixing between different basis states.

In a larger problem you would almost never print the whole Hamiltonian, but here it is worth doing once because it makes the abstract notation concrete.

In [ ]:
basis_labels = [format(i, "03b") for i in range(8)]

H = np.array(
    [
        [0.2235, -0.0390, -0.1035, -0.0818, 0.1746, 0.1091, 0.1165, -0.0104],
        [-0.0390, 0.6621, 0.0706, -0.1964, -0.0782, 0.2619, 0.1095, 0.0029],
        [-0.1035, 0.0706, 0.9961, 0.1724, 0.1067, -0.2299, -0.1817, 0.1571],
        [-0.0818, -0.1964, 0.1724, -0.1773, 0.1019, -0.4778, -0.1272, -0.0414],
        [0.1746, -0.0782, 0.1067, 0.1019, 0.1418, -0.1359, -0.1793, -0.0766],
        [0.1091, 0.2619, -0.2299, -0.4778, -0.1359, 0.1014, 0.1696, 0.0552],
        [0.1165, 0.1095, -0.1817, -0.1272, -0.1793, 0.1696, 0.4227, 0.2702],
        [-0.0104, 0.0029, 0.1571, -0.0414, -0.0766, 0.0552, 0.2702, 0.4456],
    ]
)

hamiltonian_df = pd.DataFrame(H, index=basis_labels, columns=basis_labels)
hamiltonian_df

### What to notice in the Hamiltonian table

When you look at the matrix above, you do **not** need to memorize every number.

Instead, notice these structural facts:

1. The matrix is symmetric, which is why a Hermitian eigensolver is appropriate.
2. The problem lives in an eight-dimensional space because we chose three qubits.
3. Off-diagonal couplings mean the true eigenstates will usually be superpositions rather than single basis states.

That last point is especially important. If the ground state were just one basis vector, there would be much less for SQD to discover from sampling.

## Step 4: Solve the full problem exactly

Before we approximate anything, we should know the exact answer.

Since $H$ is a real symmetric matrix, we can use a Hermitian eigensolver to compute all eigenvalues and eigenvectors. The smallest eigenvalue is the ground-state energy.

In symbols, we are looking for

$$
E_0 = \min \mathrm{spec}(H),
$$

and for the corresponding normalized eigenvector $\lvert \psi_0 \rangle$.

This exact solution will be our benchmark for judging whether the sample-based approach worked.

Said more plainly:

- the **eigenvalues** are the allowed energies of the system,
- the **lowest** eigenvalue is the ground-state energy,
- the matching eigenvector tells us how the ground state is spread across the basis states.

We do this exact solve first because SQD is an approximation method. Without a trusted reference, it would be much harder to tell whether the reduced subspace is actually good.

In [ ]:
exact_eigenvalues, exact_eigenvectors = eigh(H)
exact_ground_energy = exact_eigenvalues[0]
exact_ground_state = exact_eigenvectors[:, 0]

energy_table = pd.DataFrame(
    {
        "eigenvalue": exact_eigenvalues,
    }
)

print(f"Exact ground-state energy: {exact_ground_energy:.6f}")
energy_table

### How to read the exact output

The printed number is the best ground-state energy available for this toy Hamiltonian because it comes from the full matrix.

The table contains **all** eigenvalues, not just the smallest one. That matters because it reminds us that diagonalization gives the full energy spectrum.

For SQD, however, we mostly care about the lowest eigenvalue and its eigenvector, because that is the state whose measurement samples we will study next.

## Step 5: Inspect the exact ground state in the computational basis

Any state in this eight-dimensional Hilbert space can be expanded as

$$
\lvert \psi_0 \rangle = \sum_x a_x \lvert x \rangle,
$$

where each $x$ is a bitstring such as $011$ or $101$, and $a_x$ is the corresponding amplitude.

The measurement probability of each basis state is given by the Born rule:

$$
p(x) = |a_x|^2.
$$

If SQD is going to work, our samples need to repeatedly reveal the basis states that carry most of this probability weight.

This is one of the central ideas of the whole notebook:

- the eigenvector contains amplitude information,
- the amplitudes determine probabilities,
- the probabilities tell us which basis states show up often when we measure,
- the frequently observed states become candidates for our reduced subspace.

In [ ]:
support_df = pd.DataFrame(
    {
        "bitstring": basis_labels,
        "amplitude": exact_ground_state,
        "probability": np.abs(exact_ground_state) ** 2,
    }
).sort_values("probability", ascending=False, ignore_index=True)

support_df

### How to interpret the support table

Read the table from top to bottom.

The largest probabilities tell you where the ground state "lives" most strongly in the computational basis. Those high-probability bitstrings are exactly the ones we hope to recover from sampling.

If the probability mass were spread almost uniformly over all eight basis states, SQD would have much less of an advantage here. The method becomes attractive when important information is concentrated in a smaller part of the full space.

## Step 6: Emulate repeated measurements

In a real quantum workflow, we would prepare a state on hardware or on a simulator and then measure it many times. Each shot returns one computational basis state.

To keep this notebook focused on the SQD logic, we will emulate that measurement process directly from the exact probabilities.

If the true distribution is $p(x)$, then a finite number of shots will not reproduce it perfectly. That finite-shot noise is important because it affects which basis states we keep in our reduced subspace.

Here is the practical meaning of the code cell below:

1. We choose a number of shots.
2. We sample that many bitstrings from the exact probability distribution.
3. We count how often each bitstring appears.
4. We turn those counts into empirical probabilities.

This is the bridge between the exact state vector and the sample-based workflow.

In [ ]:
shots = 300
sampled_indices = rng.choice(len(basis_labels), size=shots, p=np.abs(exact_ground_state) ** 2)
counts = pd.Series(sampled_indices).value_counts().sort_values(ascending=False)

counts_df = pd.DataFrame(
    {
        "decimal": counts.index,
        "bitstring": [basis_labels[i] for i in counts.index],
        "count": counts.values,
    }
)
counts_df["empirical_probability"] = counts_df["count"] / shots

counts_df

### What the measurement table is telling us

The measurement table is a noisy estimate of the true probability distribution.

A beginner-friendly way to think about it is:

- the support table from the previous step shows the **true** probabilities,
- this counts table shows a **finite-sample approximation** to those probabilities.

The more shots we take, the closer the empirical table should move toward the true one on average. That is why shot count matters later when we study convergence.

## Step 7: Turn the samples into a candidate subspace

The SQD idea is simple:

- frequent bitstrings are likely to matter,
- rare or unseen bitstrings may matter less,
- so we build a reduced subspace from the configurations that appear most often.

In this toy problem the exact ground state mostly lives on two basis states, so we will keep the two most frequently observed bitstrings.

If the sampling step is informative, these two bitstrings should match the important support of the exact ground state.

In matrix language, we are now choosing the columns that will define our reduced basis. In physics language, we are deciding which measured configurations are important enough to keep.

In [ ]:
top_k = 2
selected_indices = counts_df.head(top_k)["decimal"].tolist()
selected_bitstrings = [basis_labels[i] for i in selected_indices]

B = np.eye(len(basis_labels))[:, selected_indices]

print("Selected basis states:", selected_bitstrings)
print("Shape of basis matrix B:", B.shape)

## Step 8: Project the Hamiltonian into the sampled subspace

Once we have the basis matrix $B$, the projected Hamiltonian is

$$
H_{\mathrm{sub}} = B^T H B.
$$

The overlap matrix is

$$
S = B^T B.
$$

Because the columns of $B$ are computational basis vectors, they are orthonormal, so $S = I$. Still, we compute it explicitly because the more general SQD workflow often involves a nontrivial overlap matrix.

This projection step is the mathematical heart of the approximation:

- the full Hamiltonian acts on the full eight-dimensional space,
- the basis matrix $B$ selects only the sampled directions we decided to keep,
- $B^T H B$ asks how the original Hamiltonian behaves **inside that smaller space**.

So after this step, we are no longer solving the original full problem directly. We are solving its restriction to a reduced subspace suggested by the samples.

In [ ]:
H_sub = B.T @ H @ B
S_sub = B.T @ B

print("Projected Hamiltonian:")
print(H_sub)
print()
print("Overlap matrix:")
print(S_sub)

### Why the overlap matrix is worth computing even when it is simple

In this toy example, $S$ becomes the identity because we selected ordinary computational basis vectors.

That might make the overlap matrix feel unnecessary, but it is still a good habit to include it because many practical reduced-basis methods use vectors that are **not** automatically orthonormal.

Thinking in terms of both $H_{\mathrm{sub}}$ and $S$ prepares you for the more general version of SQD used in real research workflows.

## Step 9: Solve the reduced eigenvalue problem

The reduced coefficients $c$ satisfy

$$
H_{\mathrm{sub}} c = E S c.
$$

The smallest eigenvalue of this reduced problem is our SQD energy estimate. Once we have the coefficient vector $c$, we lift it back into the original Hilbert space with

$$
\lvert \psi_{\mathrm{SQD}} \rangle = B c.
$$

This step is the payoff: a small diagonalization problem replaces the full one.

There are really two layers here:

1. We solve for the best coefficients **inside the reduced basis**.
2. We map those coefficients back to the original full space.

That second step matters because it lets us compare the SQD approximation to the exact eigenvector using the same full-space coordinates.

In [ ]:
sqd_eigenvalues, sqd_eigenvectors = eigh(H_sub, S_sub)
sqd_energy = sqd_eigenvalues[0]
sqd_coefficients = sqd_eigenvectors[:, 0]
sqd_state = B @ sqd_coefficients

print(f"SQD ground-state estimate: {sqd_energy:.6f}")
print("Reduced-space coefficients:")
print(sqd_coefficients)
print("Reconstructed state in the full basis:")
print(sqd_state)

## Step 10: Compare the SQD estimate with the exact answer

There are two natural diagnostics:

1. **Energy error**
   $$
   |E_{\mathrm{SQD}} - E_0|
   $$
2. **State overlap**
   $$
   |\langle \psi_0 \vert \psi_{\mathrm{SQD}} \rangle|
   $$

An overlap close to $1$ means the approximate state points in almost the same direction as the exact ground state.

Beginners often focus only on the energy, but the overlap is extremely useful because it tells us whether the approximate vector itself is faithful, not just whether one scalar number happened to come out close.

In [ ]:
energy_error = abs(sqd_energy - exact_ground_energy)
state_overlap = abs(np.vdot(exact_ground_state, sqd_state))

comparison_df = pd.DataFrame(
    {
        "metric": ["Exact ground energy", "SQD ground energy", "Absolute energy error", "State overlap"],
        "value": [exact_ground_energy, sqd_energy, energy_error, state_overlap],
    }
)

comparison_df

### How to judge whether the reduced problem worked

A good SQD result usually has two features at the same time:

- a small energy error,
- a large state overlap.

If the energy looks good but the overlap is poor, that can still mean the reduced subspace is missing important structure. So it is healthy to inspect more than one metric whenever possible.

## Step 11: Study how shot count and subspace size affect the answer

SQD is only as good as the subspace suggested by the samples.

Two knobs matter a lot:

- the number of measurement shots,
- the number of sampled configurations we keep.

The experiment below repeats the workflow many times and records the average energy error. This is useful because a single random draw can be lucky or unlucky.

This section answers a very practical beginner question:

**If I change the amount of data or the size of the reduced basis, how should the approximation respond?**

The experiment gives us a more trustworthy answer than one lucky run because it averages over many random trials.

In [ ]:
def run_sqd_trial(shots: int, top_k: int, seed: int) -> float:
    local_rng = np.random.default_rng(seed)
    sampled = local_rng.choice(len(basis_labels), size=shots, p=np.abs(exact_ground_state) ** 2)
    counts = pd.Series(sampled).value_counts().sort_values(ascending=False)
    chosen = counts.head(top_k).index.tolist()
    B_trial = np.eye(len(basis_labels))[:, chosen]
    H_trial = B_trial.T @ H @ B_trial
    S_trial = B_trial.T @ B_trial
    trial_energy = eigh(H_trial, S_trial)[0][0]
    return abs(trial_energy - exact_ground_energy)


shot_grid = [8, 16, 32, 64, 128, 256, 512]
subspace_grid = [1, 2, 3]
rows = []

for shots_value in shot_grid:
    for subspace_size in subspace_grid:
        errors = [
            run_sqd_trial(shots=shots_value, top_k=subspace_size, seed=1000 + rep)
            for rep in range(40)
        ]
        rows.append(
            {
                "shots": shots_value,
                "subspace_size": subspace_size,
                "mean_abs_energy_error": float(np.mean(errors)),
            }
        )

experiment_df = pd.DataFrame(rows)
experiment_df

## Step 12: Visualize the convergence trend

We expect the energy error to improve when:

- we collect more shots, because the empirical distribution better matches the true one,
- we allow a slightly larger subspace, because we reduce the chance of excluding an important basis state.

The plot below helps us see that trend instead of guessing it from a table.

When you read the figure, do not look only for a perfectly smooth curve. Sampling is noisy, so the important question is whether the overall direction is improving as the resources increase.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for subspace_size, group in experiment_df.groupby("subspace_size"):
    ax.plot(
        group["shots"],
        group["mean_abs_energy_error"],
        marker="o",
        linewidth=2,
        label=f"top_k = {subspace_size}",
    )

ax.set_xscale("log", base=2)
ax.set_xlabel("Number of shots")
ax.set_ylabel("Mean absolute energy error")
ax.set_title("How finite-shot sampling affects SQD accuracy")
ax.legend()
plt.show()

## Step 13: Show a failure case on purpose

Good examples teach success.
Great examples also teach failure.

If our subspace misses an important ground-state basis vector, the projected problem cannot recover the correct energy. We will now choose a deliberately bad subspace to see this happen in practice.

This is worth seeing because beginners sometimes think the reduced diagonalization step itself is magically fixing everything. It is not. The quality of the reduced answer depends strongly on the quality of the chosen subspace.

In [ ]:
bad_indices = [0, 3, 6]  # |000>, |011>, |110>
B_bad = np.eye(len(basis_labels))[:, bad_indices]
H_bad = B_bad.T @ H @ B_bad
S_bad = B_bad.T @ B_bad
bad_energy = eigh(H_bad, S_bad)[0][0]

print("Bad subspace bitstrings:", [basis_labels[i] for i in bad_indices])
print(f"Energy from the bad subspace: {bad_energy:.6f}")
print(f"Exact ground-state energy:   {exact_ground_energy:.6f}")

## Step 14: What this notebook should leave you with

The full SQD workflow is now visible:

1. Solve or estimate a state whose measurements reveal important basis states.
2. Sample bitstrings from that state.
3. Keep the configurations that appear important.
4. Build a reduced basis matrix $B$.
5. Form $H_{\mathrm{sub}} = B^T H B$ and $S = B^T B$.
6. Solve the reduced eigenvalue problem.
7. Check whether the answer converges as you increase shots or enrich the subspace.

In larger chemistry or materials problems, the same logic is used, but the state preparation, symmetry constraints, and Hamiltonian construction become more sophisticated.

The core lesson stays the same:

$$
\text{good samples} \longrightarrow \text{good subspace} \longrightarrow \text{good reduced diagonalization}.
$$

If you only remember one sentence from this notebook, let it be this:

**SQD does not guess the answer out of nowhere. It uses samples to decide where the important part of the full quantum problem probably lives, and then solves the physics inside that smaller region.**

## A few common beginner pitfalls to watch for

Before you leave the notebook, here are some easy mistakes to avoid when you revisit the workflow on your own:

1. Do not confuse **amplitudes** with **probabilities**. The probabilities come from squared magnitudes.
2. Do not assume a small energy error automatically means the approximate state is perfect. Check overlap too when you can.
3. Do not forget that the subspace comes from data. If the data are poor, the projected solve can only do so much.
4. Do not treat the reduced basis as arbitrary. The entire SQD idea depends on choosing it intelligently from the samples.

If those points are clear, then you have understood the main logic of the method.